In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
train_df = pd.read_csv('data/bike_sharing_train3.csv')
test_df = pd.read_csv('data/bike_sharing_test3.csv')

display(train_df)
display(test_df)

,Unnamed: 0,holiday,workingday,temp,atemp,humidity,windspeed,log_count,year,log_windspeed,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0,0,0,9.84,14.395,81,0.0000,2.833213,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,0,9.02,13.635,80,0.0000,3.713572,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0,0,9.02,13.635,80,0.0000,3.496508,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0,0,9.84,14.395,75,0.0000,2.639057,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0,0,9.84,14.395,75,0.0000,0.693147,1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,10881,0,1,15.58,19.695,50,26.0027,5.820083,0,3.295937,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10882,10882,0,1,14.76,17.425,57,15.0013,5.488938,0,2.772670,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10883,10883,0,1,13.94,15.910,61,15.0013,5.129899,0,2.772670,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10884,10884,0,1,13.94,17.425,61,6.0032,4.867534,0,1.946367,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


,Unnamed: 0,holiday,workingday,temp,atemp,humidity,windspeed,year,log_windspeed,season_1,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0,0,1,10.66,11.365,56,26.0027,1,3.295937,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,1,10.66,13.635,56,0.0000,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0,1,10.66,13.635,56,0.0000,1,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0,1,10.66,12.880,56,11.0014,1,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0,1,10.66,12.880,56,11.0014,1,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,6488,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6489,6489,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
6490,6490,0,1,10.66,12.880,60,11.0014,0,2.485023,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6491,6491,0,1,10.66,13.635,56,8.9981,0,2.302395,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [3]:
train_df.columns

Index(['Unnamed: 0', 'holiday', 'workingday', 'temp', 'atemp', 'humidity',
       'windspeed', 'log_count', 'year', 'log_windspeed', 'season_1',
       'season_2', 'season_3', 'season_4', 'weather_1', 'weather_2',
       'weather_3', 'month_1', 'month_2', 'month_3', 'month_4', 'month_5',
       'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11',
       'month_12', 'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5',
       'hour_6', 'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23'],
      dtype='str')

In [4]:
_, pvalue = pointbiserialr(train_df['holiday'], train_df['log_count'])
print(pvalue)

0.8978176602068417


In [5]:
_, pvalue = pointbiserialr(train_df['workingday'], train_df['log_count'])
print(pvalue)

0.1098184710231644


In [6]:
_, pvalue = pearsonr(train_df['temp'], train_df['log_count'])
print(pvalue)

0.0


In [7]:
_, pvalue = pearsonr(train_df['atemp'], train_df['log_count'])
print(pvalue)

0.0


In [8]:
_, pvalue = pearsonr(train_df['humidity'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [9]:
_,pvalue = pearsonr(train_df['windspeed'], train_df['log_count'])
print(f'{pvalue:.3f}')

0.000


In [10]:
_,pvalue = pointbiserialr(train_df['year'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [11]:
_, pvalue = pearsonr(train_df['log_windspeed'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [12]:
_, pvalue = pointbiserialr(train_df['season_1'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [13]:
_, pvalue = pointbiserialr(train_df['season_2'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [14]:
_, pvalue = pointbiserialr(train_df['season_3'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [15]:
_,pvalue = pointbiserialr(train_df['season_4'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0001


In [16]:
_,pvalue = pointbiserialr(train_df['weather_1'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [17]:
_,pvalue = pointbiserialr(train_df['weather_2'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0453


In [18]:
_,pvalue = pointbiserialr(train_df['weather_3'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [19]:
_,pvalue = pointbiserialr(train_df['month_1'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [20]:
_,pvalue = pointbiserialr(train_df['month_2'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [21]:
_,pvalue = pointbiserialr(train_df['month_3'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [22]:
_,pvalue = pointbiserialr(train_df['month_4'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0534


In [23]:
_,pvalue = pointbiserialr(train_df['month_5'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [24]:
_,pvalue = pointbiserialr(train_df['month_6'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [25]:
_,pvalue = pointbiserialr(train_df['month_7'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [26]:
_,pvalue = pointbiserialr(train_df['month_8'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [27]:
_,pvalue = pointbiserialr(train_df['month_9'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [28]:
_,pvalue = pointbiserialr(train_df['month_10'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.0000


In [29]:
_,pvalue = pointbiserialr(train_df['month_11'], train_df['log_count'])
print(f'{pvalue:.4f}')
#0.05보다 작긴하지만 12월을 봤을 때 증가함으로써 제거를 고려해볼만 함

0.0499


In [30]:
_,pvalue = pointbiserialr(train_df['month_12'], train_df['log_count'])
print(f'{pvalue:.4f}')

0.5049


In [31]:
for n1 in range(24) :
    _, pvalue = pointbiserialr(train_df[f'hour_{n1}'], train_df['log_count'])
    print(f'hour_{n1} pvalue : {pvalue:.4f}')

hour_0 pvalue : 0.0000
hour_1 pvalue : 0.0000
hour_2 pvalue : 0.0000
hour_3 pvalue : 0.0000
hour_4 pvalue : 0.0000
hour_5 pvalue : 0.0000
hour_6 pvalue : 0.0000
hour_7 pvalue : 0.0000
hour_8 pvalue : 0.0000
hour_9 pvalue : 0.0000
hour_10 pvalue : 0.0000
hour_11 pvalue : 0.0000
hour_12 pvalue : 0.0000
hour_13 pvalue : 0.0000
hour_14 pvalue : 0.0000
hour_15 pvalue : 0.0000
hour_16 pvalue : 0.0000
hour_17 pvalue : 0.0000
hour_18 pvalue : 0.0000
hour_19 pvalue : 0.0000
hour_20 pvalue : 0.0000
hour_21 pvalue : 0.0000
hour_22 pvalue : 0.0261
hour_23 pvalue : 0.0000


### 컬럼을 제거한다

In [32]:
a1 = ['holiday', 'workingday', 'month_4', 'month_11','month_12']
train_df.drop(a1, axis=1, inplace=True)
test_df.drop(a1, axis=1, inplace=True)

train_df.columns
#display(train_df)


Index(['Unnamed: 0', 'temp', 'atemp', 'humidity', 'windspeed', 'log_count',
       'year', 'log_windspeed', 'season_1', 'season_2', 'season_3', 'season_4',
       'weather_1', 'weather_2', 'weather_3', 'month_1', 'month_2', 'month_3',
       'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10',
       'hour_0', 'hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23'],
      dtype='str')

In [33]:
train_df.to_csv('data/bike_sharing_train4.csv')
test_df.to_csv('data/bike_sharing_test4.csv')